# Bank Marketing: Classification Models Evaluation

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


script_dir = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd()
)
data_candidates = [
    Path.cwd() / "bank-full.csv",
    script_dir / "bank-full.csv",
    Path.cwd() / "mlassignment" / "bank-full.csv",
]
data_path = next((path for path in data_candidates if path.is_file()), data_candidates[0])
df = pd.read_csv(data_path, sep=";")

X = df.drop(columns="y")
y = df["y"].map({"no": 0, "yes": 1})
if y.isna().any():
    raise ValueError("The target column contains missing or unrecognized labels.")

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_columns),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42),
    "K-Nearest Neighbor Classifier": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes Classifier (Gaussian)": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", random_state=42),
}

results = []
for model_name, model in models.items():
    model.fit(X_train_prepared, y_train)
    y_pred = model.predict(X_test_prepared)

    if hasattr(model, "predict_proba"):
        positive_class_index = list(model.classes_).index(1)
        y_score = model.predict_proba(X_test_prepared)[:, positive_class_index]
    else:
        y_score = model.decision_function(X_test_prepared)

    results.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "AUC Score": roc_auc_score(y_test, y_score),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "MCC Score": matthews_corrcoef(y_test, y_pred),
        }
    )

results_df = pd.DataFrame(results)
print(f"Dataset: {len(df):,} rows; positive subscription rate: {y.mean():.2%}")
print("Train/test split: 80/20 with stratification; random_state=42")
print(results_df.round(4).to_string(index=False))

for metric in ["Accuracy", "AUC Score", "Precision", "Recall", "F1 Score", "MCC Score"]:
    winner = results_df.loc[results_df[metric].idxmax()]
    print(f"Highest {metric}: {winner['Model']} ({winner[metric]:.4f})")

joblib.dump(
    {
        "preprocessor": preprocessor,
        "models": models,
        "feature_columns": X.columns.tolist(),
    },
    data_path.parent / "model_bundle.joblib",
)
test_data = X_test.copy()
test_data["y"] = y_test
test_data.to_csv(data_path.parent / "test_data.csv", index=False)
